In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


# Image-Conditioned Autoregressive Molecule Generation
Build preimage graphs under a fixed, immutable image graph context.


In [ ]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import warnings


In [ ]:
import networkx as nx

def _fallback_graph(extra_triangle=False, branch=False):
    g = nx.Graph()
    g.add_nodes_from([
        (0, {'label': 'C'}), (1, {'label': 'C'}), (2, {'label': 'O'}), (3, {'label': 'N'}),
        (4, {'label': 'C'}), (5, {'label': 'S'}), (6, {'label': 'C'}), (7, {'label': 'O'})
    ])
    g.add_edges_from([
        (0, 1, {'label': 'ring'}), (1, 2, {'label': 'ring'}), (2, 3, {'label': 'ring'}), (3, 0, {'label': 'ring'}),
        (2, 4, {'label': 'bridge'}), (4, 5, {'label': 'ring'}), (5, 2, {'label': 'ring'}), (3, 6, {'label': 'branch'}), (6, 7, {'label': 'branch'})
    ])
    if extra_triangle:
        g.add_node(8, label='C')
        g.add_node(9, label='N')
        g.add_edges_from([(1, 8, {'label': 'ring'}), (8, 9, {'label': 'ring'}), (9, 1, {'label': 'ring'})])
    if branch:
        g.add_node(8, label='Cl')
        g.add_edge(5, 8, label='branch')
    return g

try:
    from abstractgraph_graphicalizer.chem import PubChemLoader, draw_molecules

        loader = PubChemLoader(on_error="skip")
    assay_ids = ['2631','624249','651741','588350','463230','492952','743219','492992','463213']
    assay_id = assay_ids[1]
    size = 3000
    print(f"size: {size}")
    use_equalized = True
    limit_active = int(size // 2) if use_equalized else int(size)
    limit_inactive = int(size // 2) if use_equalized else int(size)
    graphs, targets = loader.load(
        assay_id,
        limit_active=limit_active,
        limit_inactive=limit_inactive,
    )
    targets = np.array(targets)
except Exception as exc:
    print(f'Falling back to synthetic graphs: {exc}')

    def draw_molecules(graphs, n_graphs_per_line=4):
        import matplotlib.pyplot as plt
        n = len(graphs)
        cols = min(n_graphs_per_line, max(1, n))
        rows = int(np.ceil(n / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows))
        axes = np.atleast_1d(axes).reshape(rows, cols)
        for idx, g in enumerate(graphs):
            ax = axes[idx // cols, idx % cols]
            nx.draw(g, ax=ax, with_labels=False, node_size=120)
        for idx in range(n, rows * cols):
            axes[idx // cols, idx % cols].axis('off')
        plt.tight_layout()

    graphs = [
        _fallback_graph(),
        _fallback_graph(),
        _fallback_graph(extra_triangle=True),
        _fallback_graph(extra_triangle=True),
        _fallback_graph(branch=True),
        _fallback_graph(branch=True),
    ]
    targets = np.array([0, 0, 1, 1, 0, 1])

from abstractgraph.utils import plot_graph_label_counts
_ = plot_graph_label_counts(graphs, top=20, title='Dataset info', log_scale=True)


In [ ]:
#from abstractgraph_graphicalizer.chem import draw_molecules as display_graphs
import random
import time
import networkx as nx
from collections import defaultdict
from abstractgraph.hashing import hash_graph
from abstractgraph.operators import *
from abstractgraph.display import display
from abstractgraph.graphs import graph_to_abstract_graph, graphs_to_abstract_graphs

def group_graphs_by_image_hash(graphs, decomposition_function, nbits):
    """Group graphs by the hash of their image graph."""
    buckets = {}
    for g in graphs:
        ag = graph_to_abstract_graph(
            g,
            decomposition_function=decomposition_function,
            nbits=nbits,
            label_mode='operator_hash',
        )
        img_hash = hash_graph(ag.image_graph)
        buckets.setdefault(img_hash, []).append(g)
    return buckets

In [ ]:
nbits=14
df2 = compose(name('2'), filter_by_number_of_nodes(number_of_nodes=2), tree())
df3 = compose(name('3'), filter_by_number_of_nodes(number_of_nodes=3), tree())
df4 = compose(name('4+'), filter_by_number_of_nodes(number_of_nodes=(4,100)), tree())
df5 = compose(name('cycle<5'), filter_by_number_of_nodes(number_of_nodes=(3,4)), cycle())
df6 = compose(name('cycle-5-6'), filter_by_number_of_nodes(number_of_nodes=(5,6)), cycle())
df7 = compose(name('cycle>6'), filter_by_number_of_nodes(number_of_nodes=(7,100)), cycle())
df = add(df5, df2, df3, df4, df6, df7)
decomposition_function = compose(intersection_edges(), restore_label(), df, unlabel())

buckets = group_graphs_by_image_hash(graphs, decomposition_function, nbits)
print(f"Number of buckets: {len(buckets)}")
for img_hash, bucket in buckets.items():
    if len(bucket) > 2:
        print(f"Bucket {img_hash} size: {len(bucket)}")
        draw_molecules(bucket)
        ags = graphs_to_abstract_graphs(bucket, decomposition_function, nbits)
        display(ags)

---